# 🕵️ Dia 33 — Direito e OSINT, parte 1: acesso, raspagem e termos de uso

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 33: OSINT law, part 1: access, scraping, and terms of service* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-33.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Quarta, 14 de outubro. O Dr. Hélio, jurídico da Serra Azul, trouxe três propostas da TI da cooperativa: um script de monitoramento de postagens, um "banco de vítimas" achado num fórum e um gerente que quer entrar num grupo fechado pelo celular da filha.

*"É crime? Quebra contrato? Fere a LGPD? São três perguntas diferentes."*

Neste notebook você vai:

1. **Codificar a árvore de 8 perguntas** (com o ramo LGPD).
2. **Rodar os cinco cenários do original e os três do Dr. Hélio**.
3. **Ler um `robots.txt`** com `urllib.robotparser`.
4. **Fazer o re-teste às cegas** do Passo 4.
5. **Gerar os relatos** em Markdown.

🐍 Só biblioteca padrão. Material de estudo, não orientação jurídica.

## 1. A árvore, em código

Cada cenário é descrito por respostas às perguntas. A função percorre a árvore **na ordem** e devolve o resultado, a pergunta que o disparou e o caminho percorrido. Isso atende ao Checkpoint: *todo PARE e ESCALE nomeia a pergunta específica*.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Cenario:
    nome: str
    finalidade_autorizada: bool          # Q1
    contorna_barreira: bool              # Q2
    exige_login: bool                    # Q3
    termos_proibem_metodo: bool          # Q3b
    automatizado: bool                   # Q4
    termos_ou_robots_proibem: bool       # Q4b
    dados_pessoais: bool                 # Q5
    categoria_especial: bool             # Q5b (saúde, sexual, religião, etnia, biometria, crianças)
    jurisdicao: str                      # Q6 (registre: BR/LGPD, UE/GDPR, CA/PIPEDA...)
    interage: bool                       # Q7
    condicoes: list = field(default_factory=list)


def arvore(c: Cenario):
    """Devolve (resultado, pergunta_disparadora, caminho)."""
    caminho = []
    if not c.finalidade_autorizada:
        return "PARE", "Q1 finalidade autorizada", caminho + ["Q1 não"]
    caminho.append("Q1 sim")
    if c.contorna_barreira:
        return "PARE", "Q2 barreira técnica", caminho + ["Q2 sim"]
    caminho.append("Q2 não")
    if c.exige_login:
        caminho.append("Q3 sim")
        if c.termos_proibem_metodo:
            return "ESCALE", "Q3 termos aceitos no login", caminho + ["Q3b sim"]
        caminho.append("Q3b não")
    else:
        caminho.append("Q3 não")
    if c.automatizado:
        caminho.append("Q4 sim")
        if c.termos_ou_robots_proibem:
            return "ESCALE", "Q4 automação vs termos/robots.txt", caminho + ["Q4b sim"]
        caminho.append("Q4b não (limitar taxa, identificar cliente, preferir API)")
    else:
        caminho.append("Q4 não")
    if c.dados_pessoais:
        caminho.append("Q5 sim")
        if c.categoria_especial:
            return "ESCALE", "Q5 categoria especial", caminho + ["Q5b sim"]
        caminho.append("Q5b não -> ramo de privacidade (Dia 34)")
    else:
        caminho.append("Q5 não")
    caminho.append(f"Q6 jurisdição: {c.jurisdicao}")    # registra, não decide
    if c.interage:
        return "ESCALE", "Q7 interação", caminho + ["Q7 sim"]
    caminho.append("Q7 não")
    return "SIGA COM CONDIÇÕES", "-", caminho


print("Árvore carregada. Ramo LGPD (Q6): titular no Brasil ou coleta no Brasil -> LGPD art. 3º;")
print("base legal art. 7º (sensível: art. 11); dados públicos: art. 7º §§3º-4º; art. 4º III é das autoridades.")

## 2. Os cenários

Os cinco do original (traduzidos) e os três do Dr. Hélio. As respostas abaixo são **uma** leitura razoável; o exercício é você discordar onde tiver motivo e escrever por quê.

In [ ]:
CENARIOS = [
    Cenario("1. Ler e salvar comunicados de imprensa públicos (SingleFile)",
            True, False, False, False, False, False, False, False, "local da org.", False,
            ["só passivo", "hash e hora UTC"]),
    Cenario("2. Script puxa postagens públicas sem login, 1/s; robots.txt veda a busca",
            True, False, False, False, True, True, True, False, "várias", False),
    Cenario("3. Conta pessoal busca diretório de membros por empregador e exporta",
            True, False, True, True, True, False, True, False, "várias", False),
    Cenario("4. Credenciais 'emprestadas' de membro de grupo fechado de vítimas",
            True, True, True, True, False, False, True, False, "várias", False),
    Cenario("5. Baixar banco vazado de fórum para checar telefones dos casos",
            True, False, False, False, False, False, True, True, "várias", False),
    # ➕ Dr. Hélio
    Cenario("H1. Script lê postagens públicas com 'Serra Azul' a cada minuto (X e Instagram)",
            True, False, False, False, True, True, True, False, "BR/LGPD", False),
    Cenario("H2. Baixar 'banco de vítimas do golpe do Pix' e cruzar com cooperados",
            True, False, False, False, False, False, True, True, "BR/LGPD", False),
    Cenario("H3. Gerente entra em grupo fechado de 'vendedores de painel' pelo celular da filha",
            True, True, True, True, False, False, True, False, "BR/LGPD", True),
]

primeira_rodada = {}
for c in CENARIOS:
    res, perg, cam = arvore(c)
    primeira_rodada[c.nome] = res
    print(f"\n{c.nome}\n   -> {res}" + (f"  (disparado por {perg})" if perg != "-" else ""))
    print("   caminho:", " → ".join(cam))

> 🔍 **Por que o cenário 5 e o H2 marcam "categoria especial"?** Um banco vazado de vítimas costuma misturar tudo (CPF, endereço, às vezes saúde, menores). Sem abrir, você não sabe o que tem dentro, e não deve abrir para descobrir. Na dúvida, a árvore manda ESCALE. Dependendo da jurisdição, o original lembra que pode ser PARE: a posse em si pode ser crime em alguns lugares.

> 🔍 **H3** para na **Q2** (a conta é de outra pessoa). Mesmo que não parasse, a **Q7** (interação) daria ESCALE. E põe uma pessoa da família dentro de um grupo criminoso.

## 3. A pergunta 4 na prática: `robots.txt`

O `urllib.robotparser` lê o `robots.txt` como um rastreador bem-comportado. Plano A: um site real. Plano B: um `robots.txt` embutido (fictício, no estilo dos de redes sociais).

In [ ]:
from urllib.robotparser import RobotFileParser

AGENTE = "estudo-investigador-cyber/1.0"     # identifique o cliente com honestidade (Q4)

#@markdown Site real para o plano A:
SITE = "https://www.gov.br"  #@param {type:"string"}

rp = RobotFileParser(SITE.rstrip("/") + "/robots.txt")
try:
    rp.read()                                             # baixa e interpreta
    fonte = SITE + "/robots.txt (ao vivo)"
    print("Fonte:", fonte)
    for caminho in ("/", "/anpd/pt-br", "/busca"):
        print(f"  {caminho:15s} pode? {rp.can_fetch(AGENTE, SITE + caminho)}")
    print("  crawl-delay pedido:", rp.crawl_delay(AGENTE))
except Exception as e:
    print(f"Rede indisponível ({type(e).__name__}); seguindo com o plano B.")

# Plano B: um robots.txt FICTÍCIO, embutido
ROBOTS_FICTICIO = """User-agent: *
Disallow: /search
Disallow: /explore
Allow: /about
Crawl-delay: 10
"""
rp2 = RobotFileParser()
rp2.parse(ROBOTS_FICTICIO.splitlines())
print("\nPlano B (rede-social.example, fictício):")
for caminho in ("/about", "/search?q=serra+azul", "/explore/tags/pix"):
    print(f"  {caminho:25s} pode? {rp2.can_fetch(AGENTE, 'https://rede-social.example' + caminho)}")
print("  crawl-delay:", rp2.crawl_delay(AGENTE))
print("\n➡️ /search vedado: o H1 cai em ESCALE na Q4b, como o cenário 2 do original.")
print("   Lembre: robots.txt é pedido do site, não autorização legal; e não responde Q3 nem Q5.")

## 4. 🎮 Re-teste às cegas (Passo 4)

O notebook embaralha os cenários e esconde os resultados. Responda cada um: **P** (PARE), **E** (ESCALE) ou **S** (SIGA COM CONDIÇÕES). No fim, ele compara com a primeira rodada, como o Checkpoint pede. `MODO_JOGO = False` mostra a comparação direto.

In [ ]:
import random

MODO_JOGO = True
random.seed()                                          # nova ordem a cada execução
ordem = CENARIOS[:]
random.shuffle(ordem)
CODIGO = {"P": "PARE", "E": "ESCALE", "S": "SIGA COM CONDIÇÕES"}

bate = 0
for c in ordem:
    esperado = primeira_rodada[c.nome]
    if MODO_JOGO:
        print(f"\n{c.nome}")
        r = ""
        while r not in CODIGO:
            r = input("   P = PARE | E = ESCALE | S = SIGA c/ condições: ").strip().upper()[:1]
        ok = CODIGO[r] == esperado
        bate += ok
        print("   ✅ bate" if ok else f"   ❌ não bate (primeira rodada: {esperado})")
    else:
        print(f"{c.nome} -> {esperado}")

if MODO_JOGO:
    print(f"\n🏁 {bate}/{len(ordem)} batem. O Checkpoint pede 5/5 nos cenários do original.")
    print("Onde não bateu, a árvore ou a sua leitura dos fatos precisa ficar mais clara. Anote qual.")

## 5. 📝 Os relatos, mais o seu sexto cenário

In [ ]:
#@title Seu sexto cenário { display-mode: "form" }
nome6 = "6. Monitorar o crt.sh diariamente por novos certificados com 'serraazul' no nome"  #@param {type:"string"}
finalidade = True  #@param {type:"boolean"}
barreira = False  #@param {type:"boolean"}
login = False  #@param {type:"boolean"}
termos_login_proibem = False  #@param {type:"boolean"}
automatizado = True  #@param {type:"boolean"}
termos_robots_proibem = False  #@param {type:"boolean"}
pessoais = False  #@param {type:"boolean"}
especial = False  #@param {type:"boolean"}
jurisdicao = "BR/LGPD (cliente); servidores no exterior"  #@param {type:"string"}
interage = False  #@param {type:"boolean"}

c6 = Cenario(nome6, finalidade, barreira, login, termos_login_proibem, automatizado,
             termos_robots_proibem, pessoais, especial, jurisdicao, interage,
             ["1 consulta por dia", "User-Agent identificado", "guardar só nomes de host e datas"])

linhas = ["# Relatos de cenário — Dia 33", ""]
for c in CENARIOS + [c6]:
    res, perg, cam = arvore(c)
    linhas.append(f"## {c.nome}")
    linhas.append(f"Caminho: {' → '.join(cam)}.")
    linhas.append(f"Resultado: **{res}**" + (f", disparado por {perg}." if perg != "-" else "."))
    if res == "SIGA COM CONDIÇÕES":
        linhas.append("Condições: " + (", ".join(c.condicoes) or "escreva as condições") + ".")
    else:
        linhas.append("Alternativa que chega a SIGA COM CONDIÇÕES: ______")
    linhas.append("")
texto = "\n".join(linhas)
open("relatos-dia33.md", "w", encoding="utf-8").write(texto)
print(texto)
try:
    from google.colab import files
    files.download("relatos-dia33.md")
except ImportError:
    print("(Fora do Colab: relatos-dia33.md ficou na pasta atual.)")

## ✅ Antes de fechar o Dia 33

- [ ] Árvore desenhada no diagrams.net, com o ramo LGPD na Q6
- [ ] Cinco cenários do original + H1–H3 com caminho e pergunta disparadora
- [ ] Uma alternativa para cada PARE/ESCALE
- [ ] Sexto cenário seu
- [ ] Re-teste às cegas registrado ao lado da primeira rodada

**Amanhã (Dia 34):** fechar a Fase 2. Tudo o que você coletou desde o Dia 19 tem dado pessoal em algum lugar. Registro de retenção, minimização, cifragem e descarte.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>